# nSTATPaperExamples (Python clean-room vertical slice)

This notebook executes a full spike-process workflow with deterministic validation:

1. Simulate a Poisson conditional intensity function (CIF) and spike train.
2. Fit a Poisson GLM from binned spike counts and compare recovered parameters.
3. Decode a latent discrete state sequence with point-process Bayes filtering.
4. Compute trial-wise significance using pairwise rate tests with FDR control.

Reference: Cajigas et al., *J Neurosci Methods* 2012. DOI: `10.1016/j.jneumeth.2012.08.009`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

rng = np.random.default_rng(2026)
print('Deterministic seed:', 2026)


## 1) Simulate a Poisson CIF and spike train

We use a one-dimensional stimulus covariate with a log-link Poisson model:
\(\lambda_t = \exp(\beta_0 + \beta_1 x_t)\).


In [ ]:
time = np.linspace(0.0, 20.0, 20001)
dt = float(time[1] - time[0])

stimulus = np.sin(2.0 * np.pi * 1.5 * time) + 0.3 * np.cos(2.0 * np.pi * 0.5 * time)
X = stimulus[:, None]

true_intercept = np.log(20.0)
true_beta = np.array([0.55])

true_model = CIFModel(coefficients=true_beta, intercept=true_intercept, link='poisson')
true_rate = true_model.evaluate(X)
spike_times = true_model.simulate_by_thinning(time, X, rng=rng)

print('Spikes simulated:', spike_times.size)
print('Mean true rate [Hz]:', float(np.mean(true_rate)))


## 2) Fit Poisson GLM and validate parameter recovery

In [ ]:
cov = Covariate(time=time, data=stimulus, name='stimulus', labels=['stim'])
train = SpikeTrain(spike_times=spike_times, t_start=float(time[0]), t_end=float(time[-1]), name='unit_1')
trial = Trial(spikes=SpikeTrainCollection([train]), covariates=CovariateCollection([cov]))

cfg = TrialConfig(covariate_labels=['stim'], sample_rate_hz=1.0 / dt, fit_type='poisson', name='poisson_demo')
fit = Analysis.fit_trial(trial, cfg)
est_model = fit.as_cif_model()
est_rate = est_model.evaluate(X)

coef_error = abs(float(fit.coefficients[0]) - float(true_beta[0]))
rate_rel_err = float(np.mean(np.abs(est_rate - true_rate) / np.maximum(true_rate, 1e-9)))

print('Estimated intercept:', fit.intercept)
print('Estimated beta:', float(fit.coefficients[0]))
print('Coefficient abs error:', coef_error)
print('Relative rate error:', rate_rel_err)
print('AIC:', fit.aic())
print('BIC:', fit.bic())

# Matches the parity tolerance contract for this workflow.
assert coef_error < 0.25, f'Coefficient error too high: {coef_error}'
assert rate_rel_err < 0.20, f'Relative rate error too high: {rate_rel_err}'


## 3) Decode latent state from population point-process observations

In [ ]:
n_units = 16
n_states = 21
n_time = 220

centers = np.linspace(0.0, n_states - 1, n_units)
widths = np.full(n_units, 2.4)
states_axis = np.arange(n_states)[None, :]
tuning = 0.06 + 0.45 * np.exp(-0.5 * ((states_axis - centers[:, None]) / widths[:, None]) ** 2)

transition = np.zeros((n_states, n_states), dtype=float)
for i in range(n_states):
    for j, w in ((i - 1, 0.15), (i, 0.70), (i + 1, 0.15)):
        if 0 <= j < n_states:
            transition[i, j] += w
    transition[i, :] /= np.sum(transition[i, :])

latent = np.zeros(n_time, dtype=int)
latent[0] = n_states // 2
for t in range(1, n_time):
    latent[t] = rng.choice(n_states, p=transition[latent[t - 1]])

spike_counts = np.zeros((n_units, n_time), dtype=float)
for t in range(n_time):
    spike_counts[:, t] = rng.poisson(tuning[:, latent[t]])

decoded, posterior = DecodingAlgorithms.decode_state_posterior(
    spike_counts=spike_counts,
    tuning_rates=tuning,
    transition=transition,
)

nrmse = float(np.sqrt(np.mean((decoded - latent) ** 2)) / (n_states - 1))
print('Decoding normalized RMSE:', nrmse)
assert nrmse < 0.20, f'Decoding NRMSE too high: {nrmse}'


## 4) Trial-rate significance matrix (FDR-controlled)

In [ ]:
trial_low = rng.binomial(1, 0.03, size=(6, 500)).astype(float)
trial_high = rng.binomial(1, 0.10, size=(6, 500)).astype(float)
trial_mat = np.vstack([trial_low, trial_high])

rates, pvals, sig = DecodingAlgorithms.compute_spike_rate_cis(trial_mat, alpha=0.05)
between = [sig[i, j] for i in range(6) for j in range(6, 12)]
between_rate = float(np.mean(between))

print('Mean trial rate (first 3):', rates[:3])
print('Between-group detection rate:', between_rate)
assert between_rate > 0.90, f'Between-group detection too low: {between_rate}'


## 5) Quick visualization

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=False)

axes[0].plot(time, stimulus, color='k', linewidth=1.2)
axes[0].set_title('Stimulus')
axes[0].set_ylabel('a.u.')

axes[1].plot(time, true_rate, label='true rate', linewidth=1.3)
axes[1].plot(time, est_rate, label='estimated rate', linewidth=1.1)
axes[1].set_title('Poisson CIF fit')
axes[1].set_ylabel('Hz')
axes[1].legend(loc='upper right')

axes[2].plot(latent, label='latent state', linewidth=1.2)
axes[2].plot(decoded, label='decoded state', linewidth=1.0)
axes[2].set_title('State decoding')
axes[2].set_xlabel('time bin')
axes[2].set_ylabel('state index')
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()
